# StreamView Analytics — Auditoría Etapa 1 (Persona 1)

Este notebook **no modifica los CSV originales**. Su objetivo es revisar, punto por punto, si las decisiones de limpieza de la Etapa 1 están justificadas y si el código las implementa correctamente.

## Orden de revisión
1. Carga y estructura de los datasets
2. Valores nulos
3. Duplicados
4. `duration`
5. `rating` vs `vote_average`
6. `budget` y `revenue`
7. Outliers / valores atípicos
8. Fechas y tipos de datos
9. Variables derivadas y ROI
10. Reglas de negocio
11. Validación final de los datasets limpios

> **Importante:** primero diagnosticamos y después decidimos. No vamos a aceptar automáticamente las decisiones de otro chat/IA solo porque el código funciona.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 140)

# El notebook asume que se ejecuta desde la raíz del proyecto.
ROOT = Path.cwd()
DATA = ROOT / 'data'

print('Raíz detectada:', ROOT)
print('Carpeta data:', DATA)
print('Existe data?:', DATA.exists())

## 1. Cargar los datos originales

Trabajaremos primero con los **archivos originales**, no con los limpios. Así podemos comprobar si las decisiones documentadas realmente se desprenden de los datos.

In [ ]:
MOVIES_FILE = DATA / 'netflix_movies_detailed_up_to_2025.csv'
TV_FILE = DATA / 'netflix_tv_shows_detailed_up_to_2025.csv'

print('Movies:', MOVIES_FILE, '->', MOVIES_FILE.exists())
print('TV Shows:', TV_FILE, '->', TV_FILE.exists())

movies = pd.read_csv(MOVIES_FILE)
tv = pd.read_csv(TV_FILE)

print('\nMovies:', movies.shape)
print('TV Shows:', tv.shape)

In [ ]:
print('COLUMNAS MOVIES')
print(movies.columns.tolist())

print('\nCOLUMNAS TV SHOWS')
print(tv.columns.tolist())

print('\nTIPOS MOVIES')
display(movies.dtypes.to_frame('dtype'))

print('\nTIPOS TV SHOWS')
display(tv.dtypes.to_frame('dtype'))

## 2. Valores nulos

La pauta exige tratamiento de valores nulos. Aquí distinguimos entre **cantidad** y **decisión**: encontrar un nulo no significa automáticamente que haya que eliminar la fila.

In [ ]:
def resumen_nulos(df, nombre):
    r = pd.DataFrame({
        'nulos': df.isna().sum(),
        'porcentaje': (df.isna().mean() * 100).round(2),
        'dtype': df.dtypes.astype(str)
    })
    print(nombre)
    display(r.sort_values('nulos', ascending=False))

resumen_nulos(movies, 'MOVIES — Nulos')
resumen_nulos(tv, 'TV SHOWS — Nulos')

### Pregunta de auditoría
- ¿Qué columnas tienen nulos?
- ¿Los nulos representan ausencia de información o un error?
- ¿La decisión de rellenar con `Sin dato` conserva información útil?
- ¿Hay columnas numéricas donde un nulo deba tratarse de otra forma?

## 3. Duplicados

La regla de negocio indica que cada registro debe representar un único contenido. Revisamos duplicados completos y duplicados por `show_id`.

In [ ]:
for name, df in [('Movies', movies), ('TV Shows', tv)]:
    print(f'--- {name} ---')
    print('Duplicados de fila completa:', df.duplicated().sum())
    print('Duplicados de show_id:', df['show_id'].duplicated().sum())
    print()

In [ ]:
tv_dup = tv[tv['show_id'].duplicated(keep=False)].sort_values('show_id')
print('Registros TV con show_id duplicado:')
display(tv_dup)

### Decisión pendiente
No asumir que `keep='first'` es correcto solo porque funciona. Revisar si las filas duplicadas representan el mismo contenido y si existe un criterio objetivo para elegir cuál conservar.

## 4. Auditoría de `duration`

La pauta describe `duration` como formato mixto: minutos para películas y temporadas para series. Primero verificamos qué contienen realmente los archivos.

In [ ]:
for name, df in [('Movies', movies), ('TV Shows', tv)]:
    print(f'--- {name} ---')
    print('Nulos:', df['duration'].isna().sum())
    print('Valores únicos no nulos:', df['duration'].dropna().nunique())
    display(df['duration'].value_counts(dropna=False).head(15))
    print()

### Criterio
Si una variable realmente no contiene información útil en la fuente entregada, podemos justificar su exclusión. Pero si contiene datos variados, **no debemos eliminarla simplemente porque el tratamiento sea más difícil**.

## 5. Auditoría de `rating` y `vote_average`

La pauta define `rating` como clasificación etaria y `vote_average` como valoración 0–10. Verificamos qué contienen realmente los archivos.

In [ ]:
for name, df in [('Movies', movies), ('TV Shows', tv)]:
    print(f'--- {name} ---')
    print('rating dtype:', df['rating'].dtype)
    print('vote_average dtype:', df['vote_average'].dtype)
    print('rating únicos:', df['rating'].nunique(dropna=False))
    print('vote_average únicos:', df['vote_average'].nunique(dropna=False))
    print('\nPrimeros valores de rating:')
    display(df['rating'].value_counts(dropna=False).head(20))
    print('\nComparación exacta rating == vote_average:')
    comparable = df[['rating', 'vote_average']].dropna()
    print('Filas comparables:', len(comparable))
    if len(comparable):
        print('Iguales:', (comparable['rating'].astype(float) == comparable['vote_average'].astype(float)).all())
    print()

## 6. Auditoría financiera

Esta sección es crítica porque la pauta establece que `budget` y `revenue` se usan **exclusivamente para películas** y que los contenidos sin esos datos no entran en los indicadores financieros. fileciteturn2file3L142-L152

Primero observamos los ceros, rangos y valores pequeños. **Todavía no aplicamos un umbral de $1.000.**

In [ ]:
fin = movies[['budget', 'revenue']].copy()

print('Budget = 0:', (fin['budget'] == 0).sum(), f'({(fin["budget"] == 0).mean()*100:.2f}%)')
print('Revenue = 0:', (fin['revenue'] == 0).sum(), f'({(fin["revenue"] == 0).mean()*100:.2f}%)')
print('\nBudget entre 1 y 999:', ((fin['budget'] > 0) & (fin['budget'] < 1000)).sum())

print('\nDescriptivos financieros:')
display(fin.describe().T)

print('\nPresupuestos positivos más pequeños:')
display(fin.loc[fin['budget'] > 0, 'budget'].sort_values().head(30).to_frame('budget'))

print('\nIngresos positivos más pequeños:')
display(fin.loc[fin['revenue'] > 0, 'revenue'].sort_values().head(30).to_frame('revenue'))

### Punto crítico: ¿por qué $1.000?

La regla oficial dice excluir contenidos sin datos financieros, pero **no establece un umbral de $1.000**. Por eso este notebook obliga a justificar ese criterio antes de incorporarlo al código definitivo.

Si se decide excluir valores pequeños por considerarlos anomalías, debemos dejar documentado **el criterio y su evidencia**, no solamente el número.

In [ ]:
print('Películas con budget > 0 y revenue > 0:', ((movies['budget'] > 0) & (movies['revenue'] > 0)).sum())
print('Porcentaje:', (((movies['budget'] > 0) & (movies['revenue'] > 0)).mean()*100).round(2), '%')

print('\nEjemplos con budget extremadamente pequeño y revenue positivo:')
ej = movies[(movies['budget'] > 0) & (movies['budget'] < 1000) & (movies['revenue'] > 0)]
display(ej[['show_id', 'title', 'budget', 'revenue']].sort_values('budget').head(30))

## 7. Outliers / valores atípicos

La entrega exige detección y manejo de outliers. Aquí no eliminamos automáticamente valores extremos: primero los identificamos y después decidimos si son errores o simplemente películas de gran presupuesto/ingresos.

In [ ]:
numeric_movies = movies.select_dtypes(include=np.number).columns
display(movies[numeric_movies].describe().T)

def iqr_bounds(series):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

for col in ['budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'release_year']:
    if col in movies.columns:
        low, high = iqr_bounds(movies[col])
        count = ((movies[col] < low) | (movies[col] > high)).sum()
        print(f'{col}: límites IQR=({low:.2f}, {high:.2f}) | posibles outliers={count}')

### Advertencia metodológica
Un outlier estadístico **no es automáticamente un error**. Por ejemplo, una película con ingresos extraordinariamente altos puede ser perfectamente válida. Debemos distinguir anomalías de valores extremos legítimos.

## 8. Fechas y tipos

Comprobamos `date_added`, `release_year` y posibles problemas de conversión.

In [ ]:
for name, df in [('Movies', movies), ('TV Shows', tv)]:
    print(f'--- {name} ---')
    print('date_added dtype original:', df['date_added'].dtype)
    converted = pd.to_datetime(df['date_added'], errors='coerce')
    print('fechas que no pudieron convertirse:', converted.isna().sum())
    print('release_year dtype:', df['release_year'].dtype)
    print('release_year mínimo:', df['release_year'].min())
    print('release_year máximo:', df['release_year'].max())
    print()

## 9. Variables derivadas y ROI

La pauta permite/solicita variables derivadas y establece ROI como `revenue / budget`. La validación debe comprobar que el cálculo solo se realice cuando ambos datos financieros son válidos. fileciteturn2file3L154-L162

In [ ]:
mask_fin = (movies['budget'] > 0) & (movies['revenue'] > 0)
roi = movies.loc[mask_fin, 'revenue'] / movies.loc[mask_fin, 'budget']

print('Películas con budget > 0 y revenue > 0:', mask_fin.sum())
print('ROI calculable:', roi.notna().sum())
print('\nROI descriptivo:')
display(roi.describe())

print('\nEjemplos:')
ej_roi = movies.loc[mask_fin, ['title', 'budget', 'revenue']].copy()
ej_roi['ROI'] = ej_roi['revenue'] / ej_roi['budget']
display(ej_roi.sort_values('ROI', ascending=False).head(10))

## 10. Checklist de las 9 reglas de negocio

Las reglas oficiales incluyen: un registro por contenido, comparaciones solo con variables equivalentes, finanzas solo para películas, `vote_average` 0–10, `popularity` como índice relativo, exclusión de contenidos sin datos financieros, período/alcance explícitos en visualizaciones, justificación del diseño y explicitación de variables no disponibles. fileciteturn2file3L142-L152

In [ ]:
checks = {
    '1. show_id único en Movies': movies['show_id'].is_unique,
    '1. show_id único en TV': tv['show_id'].is_unique,
    '4. vote_average dentro de 0-10 (Movies)': movies['vote_average'].between(0, 10).all(),
    '4. vote_average dentro de 0-10 (TV)': tv['vote_average'].between(0, 10).all(),
    '3/6. budget/revenue presentes solo en Movies': 'budget' in movies.columns and 'revenue' in movies.columns,
    '5. popularity no se interpreta como reproducciones': 'DOCUMENTAR EN INFORME' ,
    '7. período y alcance en gráficos': 'REVISAR CON PERSONA 2',
    '8. justificación del tipo de gráfico': 'REVISAR CON PERSONA 2',
    '9. variables no disponibles documentadas': 'REVISAR README/INFORME'
}
display(pd.DataFrame({'resultado': pd.Series(checks)}))

## 11. Validación de los archivos limpios

Después de aprobar las decisiones, esta sección comprueba los CSV finales. **No los modifica.**

In [ ]:
MOVIES_CLEAN = DATA / 'movies_limpio.csv'
TV_CLEAN = DATA / 'tv_shows_limpio.csv'

if MOVIES_CLEAN.exists() and TV_CLEAN.exists():
    movies_clean = pd.read_csv(MOVIES_CLEAN)
    tv_clean = pd.read_csv(TV_CLEAN)

    print('Movies limpio:', movies_clean.shape)
    print('TV limpio:', tv_clean.shape)

    print('\nNulos Movies limpio:')
    display(movies_clean.isna().sum().sort_values(ascending=False).head(15))

    print('\nNulos TV limpio:')
    display(tv_clean.isna().sum().sort_values(ascending=False).head(15))

    print('\nColumnas Movies limpio:')
    print(movies_clean.columns.tolist())

    print('\nColumnas TV limpio:')
    print(tv_clean.columns.tolist())
else:
    print('Aún no existen ambos archivos limpios en data/. Esta sección se ejecutará después.')

## Resultado de la auditoría

Al terminar, completar manualmente:

- [ ] Nulos: tratamiento justificado
- [ ] Duplicados: tratamiento justificado
- [ ] `duration`: decisión respaldada por los datos
- [ ] `rating`: decisión respaldada por los datos
- [ ] Finanzas: regla oficial implementada
- [ ] Umbral para valores financieros pequeños: justificado o eliminado
- [ ] Outliers: identificados y tratados con criterio
- [ ] Fechas/tipos: corregidos y validados
- [ ] ROI: calculado solo con datos financieros válidos
- [ ] Variables derivadas: verificadas
- [ ] 9 reglas de negocio: revisadas
- [ ] Dataset final: validado

### Principio de trabajo
**Primero evidencia → después decisión → después código → finalmente validación.**